In [97]:
import pandas as pd
import numpy as np

session_df = pd.read_csv("../data/analysis_table/session_table.csv")
user_df = pd.read_csv("../data/analysis_table/user_table.csv")
funnel = pd.read_csv("../data/analysis_table/funnel_table.csv")
event_df = pd.read_csv("../data/analysis_table/2019_Oct_cleaned.csv")

In [98]:
# define "Cart Abandonment": when a session has a cart event but didn't have purchase event.
abandoned = (session_df['cart_count'] > 0) & (session_df['purchase_count'] == 0)

total_sessions = len(session_df)

cart_sessions = (session_df['cart_count'] > 0).sum()
purchase_sessions = (session_df['purchase_count'] > 0).sum()
abandoned_sessions = abandoned.sum()

# calculate Cart-to-purchase rate and Abandonment rate
cart_to_purchase_rate = purchase_sessions / cart_sessions
abandonment_rate = abandoned_sessions / cart_sessions

print(cart_to_purchase_rate)

0.4435234813094834


In [99]:
# First of all, only taking the "view" events, since users see the prices in "view" events.
view_df = event_df[event_df['event_type'] == 'view'].copy()

session_price = (view_df.groupby('user_session')
                 .agg(session_avg_price = ('price', 'mean'))
                 .reset_index()
                 )

session_df = session_df.merge(session_price, on = 'user_session', how = 'left')

bins = [0, 50, 100, 500, 1000, 2000]
labels = ['0-50', '50-100', '100-500', '500-1000', '1000+']

session_df['price_bucket'] = pd.cut(session_df['session_avg_price'], bins = bins, labels = labels)

session_df[session_df['has_purchase'] > 0].head()

,Unnamed: 0,user_id,user_session,session_start,session_end,total_events,view_count,cart_count,purchase_count,total_revenue,has_view,has_cart,has_purchase,session_avg_price,price_bucket
312,312,138340325,1282680c-3083-4ace-a497-b110a13812bb,2019-11-11 05:43:32+00:00,2019-11-11 05:47:14+00:00,2,1,0,1,93.50,1,0,1,93.50,50-100
724,724,225644257,3601c672-9a75-4627-b9be-576298187981,2019-11-12 04:13:58+00:00,2019-11-12 04:20:47+00:00,4,2,1,1,40.91,1,1,1,40.91,0-50
1198,1198,253299396,e38b620b-6024-44e8-9db8-81a3a5b0e7e7,2019-11-06 10:46:09+00:00,2019-11-06 10:53:31+00:00,2,1,0,1,246.85,1,0,1,246.85,100-500
1259,1259,256164170,56bbfca9-4dab-4d43-adf7-a47ea1ae8000,2019-11-20 08:10:54+00:00,2019-11-20 08:21:14+00:00,7,5,1,1,113.23,1,1,1,69.94,50-100
1625,1625,267316896,278e8ce5-82c6-4dde-ae50-42d9bcc13b50,2019-11-14 07:17:16+00:00,2019-11-14 07:22:44+00:00,5,3,1,1,189.71,1,1,1,211.63,100-500


In [100]:
price_analysis = (session_df.groupby('price_bucket')
                  .agg(sessions = ('user_session', 'count'),
                       cart_rate = ('has_cart', 'mean'),
                       purchase_rate = ('has_purchase', 'mean')
                  ).reset_index()
                )

print(price_analysis)

/var/folders/j4/_7mlrdbn2_b0x18qdr4_qxr00000gn/T/ipykernel_5074/3119019423.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  price_analysis = (session_df.groupby('price_bucket')


  price_bucket  sessions  cart_rate  purchase_rate
0         0-50   2100718   0.125276       0.053895
1       50-100   2038757   0.115382       0.049536
2      100-500   7000553   0.132809       0.059626
3     500-1000   1880067   0.122262       0.053188
4        1000+    681833   0.112431       0.050963
